# 01 Clean  (owner: Data lead)

`data/raw.csv` -> `data/clean.csv`. Must be fully re-runnable top to bottom.
Runs in Colab (reads and writes the Drive folder `NYPA PowerHack`) or locally from the repo.
When the schema changes, update `data/DATA_CONTRACT.txt` and tell the team.

In [ ]:
from pathlib import Path
import pandas as pd

try:  # running in Colab
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/NYPA PowerHack')
except ImportError:  # running locally
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

RAW = ROOT / 'data' / 'raw.csv'
CLEAN = ROOT / 'data' / 'clean.csv'

df = pd.read_csv(RAW)
print(df.shape)
df.head()

## 1. Inspect

In [ ]:
df.info()
display(df.describe(include='all').T)
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('duplicate rows:', df.duplicated().sum())

## 2. Clean

In [ ]:
# snake_case column names
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r'[^0-9a-z]+', '_', regex=True).str.strip('_'))

# trim + lowercase text columns
for c in df.columns:
    if pd.api.types.is_string_dtype(df[c]):
        df[c] = df[c].str.strip().str.lower()

df = df.drop_duplicates()

# TODO: parse dates, e.g. df['date'] = pd.to_datetime(df['date'], errors='coerce')
# TODO: handle missing values per column (drop / impute / keep) and record it in DATA_CONTRACT.txt
# TODO: fix dtypes, outliers, bad categories

## 3. Validate and export

In [ ]:
assert not df.duplicated().any(), 'duplicates remain'
# TODO: add one assert per rule in DATA_CONTRACT.txt

df.to_csv(CLEAN, index=False)
print('wrote', CLEAN, df.shape)

After export: download `clean.csv` from Drive into the repo's `data` folder, then `git add .`, `git commit`, `git push`, and ping the team.